# 📓 08 — Registry & Model ResolutionThe unified registry is the single source of truth for robot and policy definitions.JSON-driven, hot-reloadable, and extensible at runtime.

## Architecture```registry/├── robots.json       # Robot definitions (aliases, assets, hardware)├── policies.json     # Policy providers (module, class, shorthands)├── loader.py         # JSON loading + mtime-based hot-reload├── robots.py         # Robot query/resolve/list functions├── policies.py       # Policy resolve/import/kwargs functions└── user_registry.py  # User-local overrides (~/.strands_robots/)```

In [ ]:
from strands_robots.registry import (    resolve_name, get_robot, list_robots, list_aliases,    has_sim, has_hardware, format_robot_table,)# Resolve aliases to canonical namesprint("Alias resolution:")for alias in ["franka", "SO100_follower", "g1", "panda", "so101"]:    print(f"  {alias:20s} → {resolve_name(alias)}")

In [ ]:
# List all robotsrobots = list_robots()print(f"\n{len(robots)} robots registered:")for name in robots[:10]:    info = get_robot(name)    sim_ok = "🎮" if has_sim(name) else "  "    hw_ok = "🔧" if has_hardware(name) else "  "    desc = info.get("description", "")[:50] if info else ""    print(f"  {sim_ok}{hw_ok} {name:25s} {desc}")if len(robots) > 10:    print(f"  ... and {len(robots) - 10} more")

In [ ]:
# Get full robot infoinfo = get_robot("so100")if info:    print("SO-100 robot info:")    for key, val in info.items():        print(f"  {key}: {val}")

## Policy Registry

In [ ]:
from strands_robots.registry import (    list_policy_providers, get_policy_provider, resolve_policy)print("Policy providers:")for name in list_policy_providers():    info = get_policy_provider(name)    aliases = info.get("aliases", []) if info else []    print(f"  {name:20s} aliases={aliases}")

In [ ]:
# Smart resolution: auto-detect provider from stringstest_strings = [    "mock",    "groot",    "lerobot/act_aloha_sim",    "zmq://localhost:5555",]for s in test_strings:    try:        provider, kwargs = resolve_policy(s)        print(f"  '{s}' → provider='{provider}', kwargs={kwargs}")    except Exception as e:        print(f"  '{s}' → error: {e}")

## Model Resolution`resolve_model()` finds MJCF/URDF files for simulation — checks local paths first, then Menagerie.

In [ ]:
from strands_robots.simulation.model_registry import (    resolve_model, list_available_models, register_urdf)# Resolve a robot model for simulationfor name in ["so100", "panda", "unitree_g1"]:    path = resolve_model(name)    print(f"  {name:15s} → {path}")

In [ ]:
# List all available modelsprint("\nAvailable robot models:")print(list_available_models())

## User-Local RegistryRegister custom robots that persist across sessions (stored in `~/.strands_robots/`).

In [ ]:
from strands_robots.registry import register_robot, list_user_robots, unregister_robot# Register a custom robotregister_robot(    name="my_custom_arm",    description="My custom 6-DOF arm",    category="arm",    joints=6,    aliases=["custom_arm", "mca"],)print("User robots:", list_user_robots())# Clean upunregister_robot("my_custom_arm")print("After cleanup:", list_user_robots())

## Hot-ReloadThe registry uses mtime-based caching — edit `robots.json` or `policies.json`and the changes are picked up automatically on next query (no restart needed).```pythonfrom strands_robots.registry import reloadreload()  # Force re-read of all JSON files```